# Core AI VLM Module - Vision Language Models

This notebook demonstrates how to use the `core.ai.vlm` module, which provides:
- **vlm_persist_after**: Decorator for persisting VLM analysis results to database

## Table of Contents
1. [VLM Persist Decorator](#vlm-persist-decorator)
2. [Async Functions](#async-functions)
3. [Sync Functions](#sync-functions)
4. [Custom Task Queue](#custom-task-queue)

In [1]:
import os, sys 
sys.path.append(os.path.abspath(".."))

# Import required modules
from core.ai.vlm.decorators import vlm_persist_after
from unittest.mock import Mock, MagicMock
from typing import Dict, Any

print("✅ Imports successful!")

✅ Imports successful!


## 1. VLM Persist Decorator

The `vlm_persist_after` decorator automatically persists VLM analysis results to the database after function execution.

In [2]:
# Example 1: Mock Celery task for demonstration
mock_task = Mock()
mock_task.apply_async = Mock(return_value=Mock(id="task-123"))

print("Mock task created for demonstration")

Mock task created for demonstration


In [3]:
# Example 2: Async function with vlm_persist_after
import asyncio

@vlm_persist_after(mock_task, queue="vlm")
async def predict_image_async(payload: Dict[str, Any]) -> Dict[str, Any]:
    """Async function that processes an image and returns VLM analysis"""
    # Simulate VLM processing
    return {
        "transcript": "This image shows a cat sitting on a mat",
        "prompt": "Describe this image",
        "video_metadata": {
            "width": 1920,
            "height": 1080,
            "format": "jpeg"
        },
        "video_id": "video-123",
        "frame_index": 0,
        "frame_timestamp_seconds": 0.0
    }

# Test the function
async def test_async():
    payload = {"id": "segment-456"}
    result = await predict_image_async(payload)
    
    print(f"Result: {result}")
    print(f"\nTask enqueued: {mock_task.apply_async.called}")
    if mock_task.apply_async.called:
        call_kwargs = mock_task.apply_async.call_args[1]
        print(f"Queue: {call_kwargs.get('queue')}")
        print(f"Inference payload keys: {list(call_kwargs.get('kwargs', {}).get('inference_payload', {}).keys())}")

# Run async test
await test_async()

Result: {'transcript': 'This image shows a cat sitting on a mat', 'prompt': 'Describe this image', 'video_metadata': {'width': 1920, 'height': 1080, 'format': 'jpeg', 'async_persist_task_id': 'task-123'}, 'video_id': 'video-123', 'frame_index': 0, 'frame_timestamp_seconds': 0.0}

Task enqueued: True
Queue: vlm
Inference payload keys: ['result', 'meta']


## 2. Sync Functions

The decorator also works with synchronous functions.

In [4]:
# Example: Sync function with vlm_persist_after
mock_task3 = Mock()
mock_task3.apply_async = Mock(return_value=Mock(id="task-789"))

@vlm_persist_after(mock_task3, queue="vlm")
def predict_image_sync(payload: Dict[str, Any]) -> Dict[str, Any]:
    """Sync function that processes an image"""
    return {
        "transcript": "This is a beautiful sunset over the ocean",
        "prompt": "What do you see in this image?",
        "video_metadata": {
            "time_of_day": "sunset",
            "location": "ocean"
        },
        "video_segment_id": "segment-123"  # Can also use this instead of payload.id
    }

# Test
payload = {"id": "segment-999"}
result = predict_image_sync(payload)

print(f"Result: {result}")
print(f"\nTask enqueued: {mock_task3.apply_async.called}")

Result: {'transcript': 'This is a beautiful sunset over the ocean', 'prompt': 'What do you see in this image?', 'video_metadata': {'time_of_day': 'sunset', 'location': 'ocean', 'async_persist_task_id': 'task-789'}, 'video_segment_id': 'segment-123'}

Task enqueued: True


## Summary

**vlm_persist_after decorator**:
- Automatically persists VLM analysis results to database
- Works with both async and sync functions
- Enqueues Celery tasks for async processing
- Optionally injects task ID into result

**Required Fields**:
- `transcript`: VLM-generated description
- `video_segment_id`: From payload.id or result

**Usage Pattern**:
```python
@vlm_persist_after(celery_task, queue="vlm")
async def my_vlm_function(payload):
    # Process image/video
    return {"transcript": "...", "video_segment_id": payload["id"]}
```








